# 02 — Stratified Gold Sampling (8.000 pesan)

Pilih sample stratified untuk anotasi manual. Strata: `year × match_outcome_for_player × tier`.

Target: 8.000 pesan, minimal 50 per stratum, deterministik (seed dari `configs/experiment.yaml`).

Output: `data/gold/sample.csv` + ringkasan distribusi stratum.

In [1]:
# Sel 1: Setup
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog
config = load_config('configs/experiment.yaml')
print_banner('02_gold_sampling', config)
run_log = RunLog(notebook='02_gold_sampling', config_path='configs/experiment.yaml')

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 02_gold_sampling
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: be03038
Started at: 2026-04-30T06:35:37+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: not-installed
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4


In [2]:
# Sel 2: Load processed
from src.gold.sampler import load_processed_concat

PROCESSED_ROOT = Path(config['data']['processed_root'])
GOLD_ROOT = Path(config['data']['gold_root'])
GOLD_ROOT.mkdir(parents=True, exist_ok=True)

df = load_processed_concat(PROCESSED_ROOT)
print(f'Loaded {len(df):,} pesan dari {PROCESSED_ROOT}')
print('\nDistribusi tier:')
print(df['tier'].value_counts(dropna=False))
print('\nDistribusi outcome:')
print(df['match_outcome_for_player'].value_counts(dropna=False))

Loaded 1,603,567 pesan dari data\processed

Distribusi tier:
tier
Lainnya     1475880
DPC Tour      63084
Major         37811
TI            26792
Name: count, dtype: Int64

Distribusi outcome:
match_outcome_for_player
loss       821320
win        747543
unknown     34704
Name: count, dtype: Int64


In [3]:
# Sel 3: Stratified sample
from src.gold.sampler import stratified_sample

TARGET = int(config['gold_sampling']['target_size'])
MIN_PER = int(config['gold_sampling']['min_per_stratum'])
STRATA = list(config['gold_sampling']['strata'])
SEED = int(config['seed'])

sample, stats = stratified_sample(
    df, target_size=TARGET, min_per_stratum=MIN_PER, strata_cols=STRATA, seed=SEED
)
print(f'target={stats.target_size}  actual={stats.actual_size}  strata_total={stats.n_strata}  strata_below_min={stats.n_strata_below_min}')
if stats.small_strata:
    print('\nStrata di bawah min_per_stratum:')
    for key, n in stats.small_strata[:20]:
        warn = f'[WARN] stratum={key} populasi={n} target={MIN_PER}'
        print('  ' + warn)
        run_log.add_warning(warn)
    if len(stats.small_strata) > 20:
        print(f'  ... dan {len(stats.small_strata) - 20} stratum lainnya.')

target=8000  actual=8002  strata_total=62  strata_below_min=0


In [4]:
# Sel 4: Distribusi stratum di sample
import pandas as pd
dist = sample.groupby(STRATA, observed=True).size().to_frame('n').reset_index()
print(f'Sample berisi {len(dist)} stratum unik.')
print('\nDistribusi per-tahun × outcome × tier (top 30):')
print(dist.sort_values('n', ascending=False).head(30).to_string(index=False))

Sample berisi 62 stratum unik.

Distribusi per-tahun × outcome × tier (top 30):
 year match_outcome_for_player     tier   n
 2025                     loss  Lainnya 380
 2024                     loss  Lainnya 349
 2025                      win  Lainnya 327
 2019                     loss  Lainnya 319
 2020                     loss  Lainnya 316
 2024                      win  Lainnya 305
 2019                      win  Lainnya 303
 2023                     loss  Lainnya 289
 2020                      win  Lainnya 286
 2022                     loss  Lainnya 262
 2023                      win  Lainnya 250
 2021                     loss  Lainnya 245
 2022                      win  Lainnya 242
 2021                      win  Lainnya 236
 2016                      win  Lainnya 222
 2016                     loss  Lainnya 217
 2018                     loss  Lainnya 205
 2018                      win  Lainnya 204
 2017                     loss  Lainnya 194
 2017                      win  Lainnya 

In [5]:
# Sel 5: Tulis sample.csv (kolom yang diperlukan untuk anotasi + tracing)
import hashlib
import yaml

ANNOTATION_COLS = [
    'match_id', 'time', 'player_slot', 'key',  # konten + identitas pesan
    'year', 'month', 'tier', 'leaguename', 'patch_name', 'phase',
    'match_outcome_for_player', 'duration', 'match_minute',
    # Kolom kosong yang akan diisi anotator:
]
out = sample[ANNOTATION_COLS].copy()
# Tambah kolom anotasi kosong (akan diisi anotator)
out['annotator_id'] = ''
out['sentiment'] = ''  # negative | neutral | positive
for lbl in config['labels']['toxicity_labels']:
    out[f'tox_{lbl}'] = ''  # 0/1
out['is_dota_jargon'] = ''
out['is_ambiguous'] = ''
out['notes'] = ''

sample_path = GOLD_ROOT / 'sample.csv'
out.to_csv(sample_path, index=False, encoding='utf-8')
print(f'Tertulis: {sample_path} ({len(out):,} baris, {len(out.columns)} kolom)')

# Hash file → manifest
h = hashlib.sha256(sample_path.read_bytes()).hexdigest()
print(f'sha256: {h}')
cfg_path = Path('configs/experiment.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
cfg.setdefault('data', {}).setdefault('gold_split_hash', {})['sample'] = h
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
run_log.add_output(sample_path)
run_log.add_output(cfg_path)

Tertulis: data\gold\sample.csv (8,002 baris, 24 kolom)
sha256: 018c4c3f71181b307977fceb7ba010190096f21dc2e91bd7efe1217913138c1a


In [6]:
# Sel 6: Run log
run_log.save('reports/run_log.csv')

[run_log] 02_gold_sampling → 4.83s, 2 outputs, 0 warnings → reports\run_log.csv
